In [ ]:
import pandas as pd
import altair as alt
import re
from textwrap import wrap


In [30]:
from ecostyles import EcoStyles
# Create styles instance
styles = EcoStyles()
# Register and enable a theme
styles.register_and_enable_theme(theme_name="article")  # or "article"

# Figure 1: Latest Opinion Polls

Source: https://www.whatscotlandthinks.org/questions/how-would-you-be-likely-to-use-your-constituency-vote-in-a-scottish-parliament-election-asked-since-2024-general-election-incl-reform-alba-separately/

Had to copy/paste the table into a csv

In [6]:
fig1_raw = pd.read_csv('fig1_data.csv', skiprows=1)

In [28]:
fig1_melted = fig1_raw.melt(id_vars=[fig1_raw.columns[0]], var_name='date', value_name='vote_share')
fig1_melted.columns = ['party', 'date', 'vote_share']
fig1_melted['date'] = pd.to_datetime(fig1_melted['date'].str.replace('.1', ''), format='%d-%b-%y')
fig1_melted['vote_share'] = pd.to_numeric(fig1_melted['vote_share'].str.replace('%', ''), errors='coerce') / 100
fig1_melted = fig1_melted[fig1_melted['party'] != 'Pollster']
fig1_melted

,party,date,vote_share
1,Conservative,2026-04-23,0.12
2,Labour,2026-04-23,0.18
3,Liberal Democrat,2026-04-23,0.10
4,Scottish National Party,2026-04-23,0.38
5,Reform UK,2026-04-23,0.20
...,...,...,...
391,Scottish National Party,2024-08-22,0.33
392,Reform UK,2024-08-22,0.09
393,Green,2024-08-22,0.05
394,Alba,2024-08-22,0.00


In [309]:
colour_mapping = {
    'Scottish National Party': '#FFd800', #'#FFF95D',  # Yellow
    'Conservative': '#0087dc',  # Blue
    'Labour': '#d50000',  # Red
    'Green': '#008066',  # Green
    'Reform UK': '#12B6CF',  # Lightblue
    'Liberal Democrat': '#FDBB30',  # Orange
    'Alba': '#005eba',  # Scottish Blue
    'Other': '#808080'  # Grey
}

# Adjust these values (as a fraction, e.g. 0.01 = 1pp) to nudge labels up/down
label_adjustments = {
    'Scottish National Party': 0,
    'Conservative': 0,
    'Labour': -0.01,
    'Green': 0.04,
    'Reform UK': 0,
    'Liberal Democrat': -0.01,
    'Alba': 0.00,
    'Other': 0.01,
}

# Last valid data point per party for label placement
last_points = (
    fig1_melted
    .dropna(subset=['vote_share'])
    .sort_values('date')
    .groupby('party')
    .last()
    .reset_index()
)
last_points['adjusted'] = last_points.apply(
    lambda row: row['vote_share'] + label_adjustments.get(row['party'], 0),
    axis=1
)

fig1_tooltip = [
    alt.Tooltip('party:N', title='Party'),
    alt.Tooltip('date:T', title='Date', format='%b %Y'),
    alt.Tooltip('vote_share:Q', title='Vote Share', format='.1%'),
]

lines = alt.Chart(fig1_melted).mark_line(smooth=True).transform_filter(
    'isValid(datum.vote_share)'
).encode(
    x=alt.X('date:T', axis=alt.Axis(labelFontSize=14, format='%b %Y')),
    y=alt.Y('vote_share:Q', axis=alt.Axis(labelFontSize=14, title='Share of polled support', titleFontSize=14, format='.0%')),
    color=alt.Color('party:N',
        scale=alt.Scale(domain=list(colour_mapping.keys()), range=list(colour_mapping.values())),
        legend=None
    ),
    tooltip=fig1_tooltip
)

labels = alt.Chart(last_points).mark_text(align='left', dx=5, fontSize=14).encode(
    x='date:T',
    y='adjusted:Q',
    text=alt.Text('party:N'),
    color=alt.Color('party:N',
        scale=alt.Scale(domain=list(colour_mapping.keys()), range=list(colour_mapping.values())),
        legend=None
    )
)

fig1 = (lines + labels).properties(height=300, width=500)
fig1


alt.LayerChart(...)

In [310]:
# Save to png
fig1.save('Scottish_elections_fig1.png', scale_factor=2)
# Save to json
fig1.save('Scottish_elections_fig1.json', scale_factor=2)

# Figure 2: Tax base performance gap

In [90]:
fig2_raw = pd.read_excel('FSP-Figures-what-Scotlands-finances-mean-for-the-next-parliament-February-2026H329K45.xlsx', sheet_name='Figure 1.2', skiprows=17, nrows=3)

In [155]:
fig2_melted = fig2_raw.melt(id_vars=[fig2_raw.columns[0]], var_name='year', value_name='value')
fig2_melted.columns = ['category', 'year', 'value']

def parse_fiscal_year(y):
    """Fiscal years always end in start year + 1 (e.g. 2016-17 -> 2017, 2017-28 -> 2018).
    Derive end year from start year to handle garbled suffixes."""
    if isinstance(y, pd.Timestamp):
        return str(y.year)
    s = str(y).strip()
    m = re.match(r'(\d{4})-\d{2}$', s)
    if m:
        return str(int(m.group(1)) + 1)
    m2 = re.match(r'(\d{4})$', s)
    if m2:
        return m2.group(1)
    return s

fig2_melted['year'] = pd.to_datetime(fig2_melted['year'].apply(parse_fiscal_year), format='%Y')
fig2_melted['value'] = pd.to_numeric(fig2_melted['value'], errors='coerce') * 1e6

# Fiscal year label: end year 2017 -> "2016/17"
fig2_melted['fiscal_year'] = fig2_melted['year'].apply(
    lambda d: f"'{str(d.year)[2:]}/{str(d.year + 1)[2:]}"
)
print(fig2_melted['fiscal_year'].unique())


["'17/18" "'18/19" "'19/20" "'20/21" "'21/22" "'22/23" "'23/24" "'24/25"
 "'25/26" "'26/27" "'27/28" "'28/29" "'29/30" "'30/31" "'31/32"]


In [285]:
# Version 2: single 'Income tax net position' label at projection end + solid/dashed style legend

# Ensure label/color/value_label columns are present (in case this cell runs standalone)
fig2_melted['label'] = fig2_melted['category'].apply(get_label)
fig2_melted['color'] = fig2_melted['category'].apply(get_color)
fig2_melted['value_label'] = fig2_melted['value'].apply(
    lambda v: f'£{v / 1e9:.1f}B' if pd.notna(v) else None
)

is_proj_v2 = fig2_melted['category'].str.lower().str.contains('projection')
proj_data_v2 = fig2_melted[is_proj_v2]
solid_data_v2 = fig2_melted[~is_proj_v2]

all_labels_v2 = fig2_melted[['label', 'color']].drop_duplicates().sort_values('label')
domain_v2 = all_labels_v2['label'].tolist()
color_range_v2 = all_labels_v2['color'].tolist()

# dx/dy offsets in pixels — negative dx nudges left of the last point
label_dx_v2 = {
    'Income tax net position projection': -5,
    'Policy only differences': -5,
}
label_dy_v2 = {
    'Income tax net position projection': 0,
    'Policy only differences': 0,
}
# Override display text for the projection label
display_text_v2 = {
    'Income tax net position projection': 'Income tax net position',
    'Policy only differences': 'Policy only differences',
}

# Use LAST valid point, right-aligned so text flows leftward
last_points_v2 = (
    fig2_melted.dropna(subset=['value'])
    .sort_values('year')
    .groupby('label')
    .last()
    .reset_index()
)
last_points_v2 = last_points_v2[
    last_points_v2['label'].isin(['Income tax net position projection', 'Policy only differences'])
].copy()
last_points_v2['display_label'] = last_points_v2['label'].map(display_text_v2)
last_points_v2['dx'] = last_points_v2['label'].map(label_dx_v2).fillna(-5)
last_points_v2['dy'] = last_points_v2['label'].map(label_dy_v2).fillna(0)

x_enc_v2 = alt.X('fiscal_year:O', axis=alt.Axis(labelFontSize=14, labelExpr="parseInt(slice(datum.value, 1, 3)) % 2 === 1 ? datum.value : ''"))
y_enc_v2 = alt.Y('value:Q', axis=alt.Axis(labelFontSize=14, labelExpr='"£" + datum.value / 1e9 + "B"', titleFontSize=14))
color_enc_v2 = alt.Color('label:N', scale=alt.Scale(domain=domain_v2, range=color_range_v2), legend=None)
tooltip_enc_v2 = [
    alt.Tooltip('fiscal_year:O', title='Year'),
    alt.Tooltip('label:N', title='Category'),
    alt.Tooltip('value_label:N', title='Value'),
]

dashed_v2 = alt.Chart(proj_data_v2).mark_line(strokeDash=[6, 3]).encode(
    x=x_enc_v2, y=y_enc_v2, color=color_enc_v2, tooltip=tooltip_enc_v2
)
solid_v2 = alt.Chart(solid_data_v2).mark_line().encode(
    x=x_enc_v2, y=y_enc_v2, color=color_enc_v2, tooltip=tooltip_enc_v2
)

text_layers_v2 = [
    alt.Chart(last_points_v2[last_points_v2['label'] == lbl]).mark_text(
        align='right',
        fontSize=14,
        dx=float(last_points_v2.loc[last_points_v2['label'] == lbl, 'dx'].iloc[0]),
        dy=float(last_points_v2.loc[last_points_v2['label'] == lbl, 'dy'].iloc[0]) - 7,
    ).encode(
        x='fiscal_year:O',
        y='value:Q',
        text='display_label:N',
        color=alt.Color('label:N', scale=alt.Scale(domain=domain_v2, range=color_range_v2), legend=None)
    )
    for lbl in last_points_v2['label'].unique()
]

main_v2 = alt.layer(dashed_v2, solid_v2, *text_layers_v2).resolve_scale(color='shared').properties(height=300, width=500)

# Line-style legend (solid = Outturn, dashed = Projected)
legend_lines_df = pd.DataFrame({
    'x': [0, 1, 2.5, 3.5],
    'y': [0, 0, 0, 0],
    'style': ['Outturn', 'Outturn', 'Projected', 'Projected']
})
legend_text_df = pd.DataFrame({
    'x': [1.1, 3.6],
    'y': [0, 0],
    'label': ['Outturn', 'Projected']
})

legend_line_layer = alt.Chart(legend_lines_df).mark_line(color='#888888').encode(
    x=alt.X('x:Q', axis=None, scale=alt.Scale(domain=[-0.5, 7])),
    y=alt.Y('y:Q', axis=None, scale=alt.Scale(domain=[-0.5, 0.5])),
    strokeDash=alt.StrokeDash('style:N',
        scale=alt.Scale(domain=['Outturn', 'Projected'], range=[[1, 0], [6, 3]]),
        legend=None
    )
)
legend_text_layer = alt.Chart(legend_text_df).mark_text(align='left', fontSize=14, color='#555555').encode(
    x=alt.X('x:Q', axis=None),
    y=alt.Y('y:Q', axis=None),
    text='label:N'
)

legend_v2 = (legend_line_layer + legend_text_layer).properties(width=500, height=25)

fig2_v2 = alt.vconcat(main_v2, legend_v2).configure_concat(spacing=4)
fig2_v2


alt.VConcatChart(...)

In [286]:
# Save to png
fig2_v2.save('Scottish_elections_fig2.png', scale_factor=2)
# Save to json
fig2_v2.save('Scottish_elections_fig2.json', scale_factor=2)

# Figure 3: Household disposable income

In [177]:
fig3_raw = pd.read_excel('FSP-Figures-what-Scotlands-finances-mean-for-the-next-parliament-February-2026H329K45.xlsx', sheet_name='Figure 3.1', skiprows=17, nrows=8)

In [295]:
fig3_data = fig3_raw.copy()
fig3_data.columns = ['year', 'growth_rate']

# Fiscal year label: end year 2017 -> "2016/17"
fig3_data['fiscal_year'] = fig3_data['year'].str.replace("20","'").str.replace(" 19"," '").str.replace("-","/")

# Tag bars as Outturn (historical averages) vs Projected (future years)
fig3_data['bar_type'] = fig3_data['year'].apply(
    lambda x: 'Outturn' if 'Average' in str(x) else 'Jan 2026 Forecast'
)

# Format 'Average...' labels as two lines: "Average\n'99/00-'07/08"
fig3_data['fiscal_year'] = fig3_data['fiscal_year'].apply(
    lambda x: x.replace("Average ", "Average\n").replace(" to ", "-\n") if 'Average' in str(x) else x
)
fig3_data['growth_rate_label'] = fig3_data['growth_rate'].apply(lambda v: f'{v:.1f}%' if pd.notna(v) else '')


In [296]:
# Preserve left-to-right order from source data
fig3_sort_order = fig3_data['fiscal_year'].tolist()

fig3 = alt.Chart(fig3_data).mark_bar().encode(
    x=alt.X('fiscal_year:O',
        sort=fig3_sort_order,
        axis=alt.Axis(
            labelFontSize=13,
            labelAngle=0,
            labelLimit=200,
            labelExpr="split(datum.value, '\\n')",
        )
    ),
    y=alt.Y('growth_rate:Q', axis=alt.Axis(labelFontSize=14, labelExpr='datum.value + "%"')),
    opacity=alt.Opacity('bar_type:N',
        scale=alt.Scale(domain=['Outturn', 'Jan 2026 Forecast'], range=[1.0, 0.4]),
        legend=alt.Legend(title='', orient='bottom', labelFontSize=13, symbolType='square', symbolSize=150)
    ),
    tooltip=[
        alt.Tooltip('fiscal_year:O', title='Fiscal Year'),
        alt.Tooltip('growth_rate_label', title='Growth Rate'),
        alt.Tooltip('bar_type:N', title='Type'),
    ]
).properties(height=300, width=500)
fig3


alt.Chart(...)

In [297]:
# Save to png
fig3.save('Scottish_elections_fig3.png', scale_factor=2)
# Save to json
fig3.save('Scottish_elections_fig3.json', scale_factor=2)

# Figure 4: Population growth in Scotland over the current and next parliament (by age group)

In [241]:
fig4_raw = pd.read_excel('FSP-Figures-what-Scotlands-finances-mean-for-the-next-parliament-February-2026H329K45.xlsx', sheet_name='Figure 3.2', skiprows=17, nrows=4)

In [252]:
fig4_melted = fig4_raw.melt(id_vars=[fig4_raw.columns[0]], var_name='year', value_name='value')

In [253]:
fig4_melted.columns = ['category', 'year', 'value']
fig4_melted['year'] = pd.to_datetime(fig4_melted['year'], format='%Y')
fig4_melted['value'] = pd.to_numeric(fig4_melted['value'], errors='coerce')

In [298]:
fig4_colour_mapping = {
    'Under 16':    '#d50000',
    '16-64':       '#0087dc',
    '65 and over': '#008066',
    'Total':       '#808080',
}

fig4_label_adjustments = {
    'Under 16':    0,
    '16-64':       0,
    '65 and over': 0,
    'Total':       0,
}

fig4_last = (
    fig4_melted
    .dropna(subset=['value'])
    .sort_values('year')
    .groupby('category')
    .last()
    .reset_index()
)
fig4_last['adjusted'] = fig4_last.apply(
    lambda row: row['value'] + fig4_label_adjustments.get(row['category'], 0),
    axis=1
)

lines = alt.Chart(fig4_melted).mark_line().encode(
    x=alt.X('year:T', axis=alt.Axis(labelFontSize=14, format='%Y')),
    y=alt.Y('value:Q',
        scale=alt.Scale(domain=[80, 130]),
        axis=alt.Axis(labelFontSize=14, title='Population indexed to 2021=100', titleFontSize=14)
    ),
    color=alt.Color('category:N',
        scale=alt.Scale(domain=list(fig4_colour_mapping.keys()), range=list(fig4_colour_mapping.values())),
        legend=None
    ),
    tooltip=[
        alt.Tooltip('category:N', title='Group'),
        alt.Tooltip('year:T', title='Year', format='%Y'),
        alt.Tooltip('value:Q', title='Index', format='.1f'),
    ]
)

fig4_labels = alt.Chart(fig4_last).mark_text(align='left', dx=5, fontSize=14).encode(
    x='year:T',
    y=alt.Y('adjusted:Q', scale=alt.Scale(domain=[80, 130])),
    text='category:N',
    color=alt.Color('category:N',
        scale=alt.Scale(domain=list(fig4_colour_mapping.keys()), range=list(fig4_colour_mapping.values())),
        legend=None
    )
)

# Horizontal dashed reference line at 100
baseline_df = pd.DataFrame({'y': [100]})
baseline = alt.Chart(baseline_df).mark_rule(
    strokeDash=[4, 4], color='black', strokeWidth=1
).encode(y=alt.Y('y:Q', scale=alt.Scale(domain=[80, 130])))

fig4 = (baseline + lines + fig4_labels).properties(height=300, width=450)
fig4


alt.LayerChart(...)

In [299]:
# Save to png
fig4.save('Scottish_elections_fig4.png', scale_factor=2)
# Save to json
fig4.save('Scottish_elections_fig4.json', scale_factor=2)